In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Configuración
input_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW"
salida_dir = r"M:\material_isa\estadisticos\_Turberas\graficas_comparativas"
os.makedirs(salida_dir, exist_ok=True)

# Paleta de colores fijos
colores = {
    "Pleta_de_Molieres": "#8c564b", # marrón
    "Clots_de_Rialba": "#ff7f0e",  # naranja
    "Estanyeres": "#2ca02c",       # verde
    "Bassa_Nera": "#1f77b4",       # azul
    "Plan_deth_Horn": "#d62728",   # rojo
    "Pla_de_Sartari": "#9467bd",    # morado
    
}

# Formatos de fecha
def parse_fecha(fecha_str, nombre):
    if nombre == "Estanyeres":
        return datetime.strptime(fecha_str.strip(), "%d/%m/%Y")
    else:
        return datetime.strptime(fecha_str.strip(), "%Y-%m-%d")

# Función para graficar un índice para todas las turberas
def graficar_todas_las_turberas(indice, nombre_salida):
    plt.figure(figsize=(14, 6))
    for file in os.listdir(input_dir):
        if file.endswith("_sin_periodo_nieve.csv"):
            nombre = file.replace("_sin_periodo_nieve.csv", "")
            df = pd.read_csv(os.path.join(input_dir, file), sep=";", encoding="latin1")
            if indice not in df.columns:
                continue

            df["FECHA"] = df["FECHA"].astype(str).str.strip()
            df = df[df["FECHA"] != ""]
            try:
                df["FECHA_dt"] = df["FECHA"].apply(lambda x: parse_fecha(x, nombre))
            except Exception as e:
                print(f"❌ Error en fechas para {nombre}: {e}")
                continue
            df = df.dropna(subset=["FECHA_dt", indice])

            plt.scatter(df["FECHA_dt"], df[indice], s=8, label=nombre.replace("_", " "), color=colores.get(nombre, None), alpha=0.7)

    plt.title(f"{indice} – Todas las Turberas")
    plt.xlabel("Fecha")
    plt.ylabel(indice)
    plt.ylim(0, 1)
    plt.grid(True)
    plt.legend(title="Turbera", loc="best")
    plt.tight_layout()

    path_final = os.path.join(salida_dir, f"{nombre_salida}.png")
    plt.savefig(path_final, dpi=150)
    plt.close()
    print(f"✅ Guardado: {path_final}")

# Ejecutar para los dos índices deseados
graficar_todas_las_turberas("NDVI", "NDVI_todas_turberas")
graficar_todas_las_turberas("NDWI_v", "NDWI_v_todas_turberas")

# --------------------- graficos lineas de regresion
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from statsmodels.nonparametric.smoothers_lowess import lowess

# --- CONFIGURACIÓN ---
input_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW"
salida_dir = r"M:\material_isa\estadisticos\_Turberas\graficas_comparativas"

turberas = {
    "Bassa_Nera": "Bassa Nera",
    "Clots_de_Rialba": "Clots de Rialba",
    "Estanyeres": "Estanyeres",
    "Plan_deth_Horn": "Plan deth Horn",
    "Pla_de_Sartari": "Pla de Sartari",
    "Pleta_de_Molieres": "Pleta de Molieres"
}

colores = {
    "Bassa_Nera": "#1f77b4",
    "Clots_de_Rialba": "#ff7f0e",
    "Estanyeres": "#2ca02c",
    "Plan_deth_Horn": "#d62728",
    "Pla_de_Sartari": "#9467bd",
    "Pleta_de_Molieres": "#8c564b"
}

# --- FUNCIÓN PARA CARGAR Y PARSEAR CSV ---
def cargar_turbera(nombre, fecha_formato):
    archivo = os.path.join(input_dir, f"{nombre}_sin_periodo_nieve.csv")
    df = pd.read_csv(archivo, sep=";", encoding="latin1", decimal=",")
    df.columns = [c.strip().replace('\ufeff', '') for c in df.columns]

    df["FECHA"] = df["FECHA"].astype(str).str.strip()
    df = df[df["FECHA"] != ""]
    df["FECHA_dt"] = pd.to_datetime(df["FECHA"], format=fecha_formato, errors="coerce")
    return df.dropna(subset=["FECHA_dt"])

# --- FUNCIÓN PARA GRAFICAR CON LOWESS ---
from statsmodels.nonparametric.smoothers_lowess import lowess
from matplotlib.dates import date2num

def graficar_con_regresion(indice, nombre_archivo_out, ylim):
    plt.figure(figsize=(14, 6))
    for file in os.listdir(input_dir):
        if file.endswith("_sin_periodo_nieve.csv"):
            nombre = file.replace("_sin_periodo_nieve.csv", "")
            if nombre not in colores:
                continue

            df = pd.read_csv(os.path.join(input_dir, file), sep=';', encoding='latin1')
            if indice not in df.columns:
                continue

            df["FECHA"] = df["FECHA"].astype(str).str.strip()
            df = df[df["FECHA"] != ""]
            try:
                df["FECHA_dt"] = df["FECHA"].apply(lambda x: parse_fecha(x, nombre))
            except Exception as e:
                print(f"❌ Error en fechas para {nombre}: {e}")
                continue
            df = df.dropna(subset=["FECHA_dt", indice])

            x = df["FECHA_dt"]
            y = df[indice]
            color = colores[nombre]

            # Puntos originales
            plt.scatter(x, y, label=nombre.replace("_", " "), alpha=0.5, s=15, color=color)

            # LOWESS con fechas convertidas a número
            x_num = date2num(x)
            lowess_smoothed = lowess(y, x_num, frac=0.2)
            plt.plot(lowess_smoothed[:, 0], lowess_smoothed[:, 1], '-', linewidth=1.5, color=color)


    plt.title(f"{indice} – Todas las Turberas (con regresión LOWESS)")
    plt.xlabel("Fecha")
    plt.ylabel(indice)
    plt.ylim(*ylim)
    plt.grid(True)
    plt.legend(title="Turbera", loc="best")
    plt.tight_layout()

    salida = os.path.join(salida_dir, nombre_archivo_out)
    plt.savefig(salida, dpi=150)
    plt.close()
    print(f"✅ Guardado: {salida}")


# --- EJECUCIÓN PARA NDVI Y NDWI_v ---
graficar_con_regresion("NDVI", "NDVI_todas_turberas_regresion.png", (0.0, 1.0))
graficar_con_regresion("NDWI_v", "NDWI_v_todas_turberas_regresion.png", (0.0, 1.0))

print("\n🎯 Finalizado: gráficos con regresión LOWESS generados.")





✅ Guardado: M:\material_isa\estadisticos\_Turberas\graficas_comparativas\NDVI_todas_turberas_regresion.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\graficas_comparativas\NDWI_v_todas_turberas_regresion.png

🎯 Finalizado: gráficos con regresión LOWESS generados.
